# COMPSS 211 — Week 2
## Python for Data Work

This notebook uses a small collection of fictional social-media posts to review:

- Python objects
- Pandas filtering and derived columns
- Missing values
- Grouped summaries
- Small reusable functions
- Debugging and verification

The dataset is created inside the notebook, so no download is required.

In [ ]:
import pandas as pd
import numpy as np

posts = [
    {
        "post_id": 1,
        "author": "user_101",
        "platform": "forum",
        "text": "Public transit should be free.",
        "likes": 27,
        "verified": False,
        "region": "West"
    },
    {
        "post_id": 2,
        "author": "user_102",
        "platform": "forum",
        "text": "The city needs more protected bike lanes.",
        "likes": 42,
        "verified": True,
        "region": "West"
    },
    {
        "post_id": 3,
        "author": "user_103",
        "platform": "microblog",
        "text": None,
        "likes": 5,
        "verified": False,
        "region": "South"
    },
    {
        "post_id": 4,
        "author": "user_104",
        "platform": "microblog",
        "text": "Remote work has changed downtown neighborhoods.",
        "likes": 31,
        "verified": True,
        "region": "Northeast"
    },
    {
        "post_id": 5,
        "author": "user_105",
        "platform": "forum",
        "text": "Libraries are essential public infrastructure.",
        "likes": 18,
        "verified": False,
        "region": "South"
    },
    {
        "post_id": 6,
        "author": "user_106",
        "platform": "microblog",
        "text": "",
        "likes": 2,
        "verified": False,
        "region": "West"
    },
]

df = pd.DataFrame(posts)
df

## 1. Python objects behind a dataframe

A dataframe is built from familiar Python objects. Here, `posts` is a list, and each item in the list is a dictionary.

In [ ]:
type(posts), type(posts[0])

In [ ]:
first_post = posts[0]

print(first_post["author"])
print(first_post["text"])
print(first_post["likes"])

### Nested access

In API work, you will often receive lists of dictionaries. You need to know how to inspect the structure before turning it into a dataframe.

In [ ]:
example_response = {
    "results": [
        {"id": 10, "title": "Article A", "topics": ["policy", "cities"]},
        {"id": 11, "title": "Article B", "topics": ["labor"]}
    ],
    "count": 2
}

print(example_response["count"])
print(example_response["results"][0]["title"])
print(example_response["results"][0]["topics"][1])

## 2. Inspect before transforming

Before editing a dataframe, check its shape, columns, data types, and missing values.

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())

## 3. Filtering rows

Use Boolean conditions to select the rows you need.

In [ ]:
popular_posts = df[df["likes"] >= 20]
popular_posts

In [ ]:
popular_verified_posts = df[
    (df["likes"] >= 20) &
    (df["verified"] == True)
]

popular_verified_posts

### Important

With multiple conditions, place each condition inside parentheses and use:

- `&` for AND
- `|` for OR
- `~` for NOT

## 4. Creating derived columns

A derived column is calculated from existing data.

In [ ]:
df["word_count"] = df["text"].fillna("").str.split().str.len()
df["is_popular"] = df["likes"] >= 20

df[["post_id", "text", "likes", "word_count", "is_popular"]]

## 5. Missing and unusable text

`None`, `NaN`, and empty strings are not identical. For text analysis, we often need to remove both missing and empty text.

In [ ]:
usable_text = df[
    df["text"].notna() &
    (df["text"].str.strip() != "")
].copy()

usable_text

## 6. Grouped summaries

A grouped summary changes the unit of analysis. The result below has one row per platform, not one row per post.

In [ ]:
platform_summary = (
    usable_text
    .groupby("platform")
    .agg(
        number_of_posts=("post_id", "count"),
        average_likes=("likes", "mean"),
        average_words=("word_count", "mean")
    )
    .reset_index()
)

platform_summary

## 7. Turning notebook steps into a function

A function makes a workflow reusable. It should have clear inputs and outputs.

In [ ]:
def prepare_posts(posts_df):
    cleaned = posts_df.copy()

    cleaned = cleaned[
        cleaned["text"].notna() &
        (cleaned["text"].str.strip() != "")
    ].copy()

    cleaned["text_lower"] = cleaned["text"].str.lower()
    cleaned["word_count"] = cleaned["text"].str.split().str.len()
    cleaned["is_popular"] = cleaned["likes"] >= 20

    return cleaned

In [ ]:
cleaned_df = prepare_posts(df)
cleaned_df

## 8. Verification

Code running without an error does not guarantee that the result is correct.

Check:

- Did the number of rows change as expected?
- Are the expected columns present?
- Are data types sensible?
- Are there remaining missing values?
- Do a few sample rows look correct?

In [ ]:
print("Original rows:", len(df))
print("Cleaned rows:", len(cleaned_df))
print("Columns:", cleaned_df.columns.tolist())
print("Missing text values:", cleaned_df["text"].isna().sum())

assert len(cleaned_df) == 4
assert "word_count" in cleaned_df.columns
assert cleaned_df["text"].isna().sum() == 0

cleaned_df.head()

## 9. Debugging common errors

Run each example one at a time. Read the final line of the traceback before changing the code.

In [ ]:
# Example 1: KeyError
# Uncomment the line below.

# df["like_count"]

In [ ]:
# Example 2: TypeError
# Uncomment the line below.

# "Likes: " + df.loc[0, "likes"]

In [ ]:
# Example 3: AttributeError
# Uncomment the line below.

# df["likes"].lower()

### A debugging routine

1. Read the final line of the traceback.
2. Identify the failing line.
3. Inspect the object and its type.
4. Reduce the problem to a smaller example.
5. Check what output you expected.
6. Make one change at a time.

## 10. Exit prompt

In two or three sentences, explain:

1. What one row represents in `cleaned_df`.
2. What `prepare_posts()` expects as input.
3. What it returns.
4. One assumption the function makes.